In [ ]:
#============================================================
# Experiment 12A: AdaBoost Classification from Scratch
# Binary Classification - Weak Learners: Decision Stumps
#============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Create Synthetic Dataset
# X1 = Study Hours, X2 = Attendance
# Class -1 = Fail, Class +1 = Pass
X = np.array([
    [1.0, 55], [1.5, 58], [2.0, 60], [2.5, 63],
    [3.0, 65], [3.5, 67], [4.0, 70], [4.5, 72],
    [5.0, 75], [5.5, 78], [6.0, 80], [6.5, 82],
    [7.0, 85], [7.5, 88], [8.0, 90], [8.5, 92]
], dtype=float)

y = np.array([-1, -1, -1, -1, -1, -1, -1, -1,
               1,  1,  1,  1,  1,  1,  1,  1])

# Step 2: Train-Test Split From Scratch
np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.80 * len(X))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train = X[train_indices]
y_train = y[train_indices]
X_test = X[test_indices]
y_test = y[test_indices]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# Step 3: Decision Stump (Weak Learner)
class DecisionStump:

    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1

    def predict(self, X):
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)

        if self.polarity == 1:
            predictions[X[:, self.feature_index] < self.threshold] = -1
        else:
            predictions[X[:, self.feature_index] >= self.threshold] = -1

        return predictions

# Step 4: AdaBoost Classifier
class AdaBoost:

    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.stumps = []
        self.alphas = []
        self.errors = []

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize Equal Sample Weights
        weights = np.ones(n_samples) / n_samples

        # Train Weak Learners
        for iteration in range(self.n_estimators):
            stump = DecisionStump()
            best_error = float("inf")

            # Search for Best Feature, Threshold and Polarity
            for feature in range(n_features):
                feature_values = np.unique(X[:, feature])

                for threshold in feature_values:
                    for polarity in [1, -1]:
                        stump_test = DecisionStump()
                        stump_test.feature_index = feature
                        stump_test.threshold = threshold
                        stump_test.polarity = polarity

                        predictions = stump_test.predict(X)

                        # Calculate Weighted Error
                        incorrect = (predictions != y)
                        error = np.sum(weights[incorrect])

                        if error < best_error:
                            best_error = error
                            stump = stump_test

            # Calculate Alpha (Learner Weight)
            epsilon = 1e-10
            alpha = 0.5 * np.log((1 - best_error + epsilon) / (best_error + epsilon))

            # Get Predictions
            predictions = stump.predict(X)

            # Update Sample Weights
            weights *= np.exp(-alpha * y * predictions)

            # Normalize Weights
            weights /= np.sum(weights)

            # Store Model Information
            self.stumps.append(stump)
            self.alphas.append(alpha)
            self.errors.append(best_error)

            print(f"Iteration {iteration+1}: Error={best_error:.4f}, Alpha={alpha:.4f}")

    def predict(self, X):
        n_samples = X.shape[0]
        final_prediction = np.zeros(n_samples)

        # Weighted Combination of Weak Learners
        for stump, alpha in zip(self.stumps, self.alphas):
            prediction = stump.predict(X)
            final_prediction += alpha * prediction

        return np.sign(final_prediction)

# Step 5: Create AdaBoost Model
model = AdaBoost(n_estimators=10)

# Step 6: Train Model
print("\n===================================")
print("TRAINING ADABOOST")
print("===================================")
model.fit(X_train, y_train)

# Step 7: Make Predictions
y_pred = model.predict(X_test)

# Step 8: Display Predictions
print("\nActual vs Predicted")
print("==============================")
print("Actual\tPredicted")
for actual, predicted in zip(y_test, y_pred):
    print(f"{int(actual)}\t{int(predicted)}")

# Step 9: Calculate Accuracy
accuracy = np.mean(y_pred == y_test)

print("\n===================================")
print("ADABOOST PERFORMANCE")
print("===================================")
print("Correct Predictions:", np.sum(y_pred == y_test))
print("Total Predictions:", len(y_test))
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 10: Confusion Matrix From Scratch
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == -1) & (y_pred == -1))
FP = np.sum((y_test == -1) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == -1))

print("\nConfusion Matrix")
print("-------------------------")
print("Predicted")
print("         -1  +1")
print(f"Actual -1 {TN:3d} {FP:3d}")
print(f"Actual +1 {FN:3d} {TP:3d}")

# Steps 11-13: Precision, Recall, F1 Score
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
f1 = (2 * precision * recall / (precision + recall)
      if (precision + recall) != 0 else 0)

print("\nClassification Metrics")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

# Step 14: Display Weak Learner Information
print("\n===================================")
print("WEAK LEARNERS")
print("===================================")
for i, (stump, alpha, error) in enumerate(
        zip(model.stumps, model.alphas, model.errors)):
    print(f"\nStump {i+1}")
    print("Feature:", stump.feature_index)
    print("Threshold:", stump.threshold)
    print("Polarity:", stump.polarity)
    print("Error:", round(error, 4))
    print("Alpha:", round(alpha, 4))

# Step 15: Predict New Student
new_student = np.array([[5.0, 76]])
new_prediction = model.predict(new_student)[0]

print("\n===================================")
print("NEW STUDENT PREDICTION")
print("===================================")
print("Study Hours:", new_student[0, 0])
print("Attendance:", new_student[0, 1])
print("Predicted Class:", int(new_prediction))
if new_prediction == 1:
    print("Result: PASS")
else:
    print("Result: FAIL")

# Step 16: Plot Training Error
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(model.errors) + 1), model.errors, marker="o")
plt.xlabel("Boosting Iteration")
plt.ylabel("Weighted Error")
plt.title("AdaBoost: Weak Learner Error")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 17: Plot Alpha Values
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(model.alphas) + 1), model.alphas, marker="o")
plt.xlabel("Boosting Iteration")
plt.ylabel("Alpha")
plt.title("AdaBoost: Weak Learner Weights")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 18: Plot Dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == -1, 0], X[y == -1, 1], label="Fail", marker="o")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Pass", marker="x")
plt.scatter(new_student[0, 0], new_student[0, 1],
            marker="*", s=180, label="New Student")
plt.xlabel("Study Hours")
plt.ylabel("Attendance (%)")
plt.title("AdaBoost Classification")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Experiment 12B: AdaBoost Classification using Scikit-learn
# Real-World Dataset: Breast Cancer Wisconsin Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset Shape:")
print(X.shape)
print("\nTarget Classes:")
print(target_names)

# Step 2: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map({0: target_names[0], 1: target_names[1]})

print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nClass Distribution:")
print(df["Class"].value_counts())
print("\nDataset Description:")
print(df.describe())

# Step 4: Separate Features and Target
X = df[feature_names].values
y = df["Target"].values

# Step 5: Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 6: Create Weak Learner (Decision Stump)
base_model = DecisionTreeClassifier(max_depth=1, random_state=42)

# Step 7: Create AdaBoost Model
model = AdaBoostClassifier(
    estimator=base_model,
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

# Step 8: Train AdaBoost Model
model.fit(X_train, y_train)

# Step 9: Make Predictions
y_pred = model.predict(X_test)

# Step 10: Prediction Probabilities
y_probability = model.predict_proba(X_test)

# Step 11: Display Actual and Predicted Values
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability_Class_0": y_probability[:, 0],
    "Probability_Class_1": y_probability[:, 1]
})
results["Actual Class"] = results["Actual"].map({0: target_names[0], 1: target_names[1]})
results["Predicted Class"] = results["Predicted"].map({0: target_names[0], 1: target_names[1]})

print("\nActual vs Predicted:")
print(results.head(10))

# Step 12: Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n===================================")
print("ADABOOST PERFORMANCE")
print("===================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 13: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Step 14: Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Step 15: Display Number of Weak Learners
print("\nNumber of Weak Learners:")
print(len(model.estimators_))

# Step 16: Display Estimator Weights
print("\nWeak Learner Weights:")
print(model.estimator_weights_)

# Step 17: Display Estimator Errors
print("\nWeak Learner Errors:")
print(model.estimator_errors_)

# Step 18: Feature Importance
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})
importance = importance.sort_values("Importance", ascending=False)

print("\nFeature Importance:")
print(importance)

# Step 19: Top 10 Important Features
top_features = importance.head(10)
print("\nTop 10 Important Features:")
print(top_features)

# Step 20: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("AdaBoost Feature Importance")
plt.tight_layout()
plt.show()

# Step 21: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("AdaBoost Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names)
plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

# Step 22: Actual vs Predicted Plot
sample_number = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(sample_number, y_test, label="Actual", marker="o")
plt.scatter(sample_number, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number")
plt.ylabel("Class")
plt.title("AdaBoost: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 23: Accuracy During Boosting Iterations
staged_accuracy = []
for prediction in model.staged_predict(X_test):
    staged_accuracy.append(accuracy_score(y_test, prediction))

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(staged_accuracy) + 1), staged_accuracy, marker="o")
plt.xlabel("Boosting Iteration")
plt.ylabel("Test Accuracy")
plt.title("AdaBoost: Accuracy vs Boosting Iteration")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 24: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_prediction = model.predict(new_sample)[0]
new_probability = model.predict_proba(new_sample)[0]

print("\n===================================")
print("NEW SAMPLE PREDICTION")
print("===================================")
print("Predicted Class:", target_names[new_prediction])
print("Probability of Class 0:", round(new_probability[0], 4))
print("Probability of Class 1:", round(new_probability[1], 4))

In [ ]:
#============================================================
# Experiment 12C: Gradient Boosting Classification using Scikit-learn
# Real-World Dataset: Breast Cancer Wisconsin Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset Shape:")
print(X.shape)
print("\nTarget Classes:")
print(target_names)

# Step 2: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map({0: target_names[0], 1: target_names[1]})

print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nClass Distribution:")
print(df["Class"].value_counts())
print("\nDataset Description:")
print(df.describe())

# Step 4: Separate Features and Target
X = df[feature_names].values
y = df["Target"].values

# Step 5: Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 6: Create Gradient Boosting Model
model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Step 7: Train the Model
model.fit(X_train, y_train)

# Step 8: Make Predictions
y_pred = model.predict(X_test)

# Step 9: Prediction Probabilities
y_probability = model.predict_proba(X_test)

# Step 10: Display Actual and Predicted Values
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability_Class_0": y_probability[:, 0],
    "Probability_Class_1": y_probability[:, 1]
})
results["Actual Class"] = results["Actual"].map({0: target_names[0], 1: target_names[1]})
results["Predicted Class"] = results["Predicted"].map({0: target_names[0], 1: target_names[1]})

print("\nActual vs Predicted:")
print(results.head(10))

# Step 11: Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n===================================")
print("GRADIENT BOOSTING PERFORMANCE")
print("===================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 12: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Step 13: Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Step 14: Feature Importance
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})
importance = importance.sort_values("Importance", ascending=False)

print("\nFeature Importance:")
print(importance)

# Step 15: Top 10 Important Features
top_features = importance.head(10)
print("\nTop 10 Important Features:")
print(top_features)

# Step 16: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Gradient Boosting Feature Importance")
plt.tight_layout()
plt.show()

# Step 17: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Gradient Boosting Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names)
plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

# Step 18: Actual vs Predicted Plot
sample_number = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(sample_number, y_test, label="Actual", marker="o")
plt.scatter(sample_number, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number")
plt.ylabel("Class")
plt.title("Gradient Boosting: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 19: Accuracy During Boosting Iterations
staged_accuracy = []
for prediction in model.staged_predict(X_test):
    staged_accuracy.append(accuracy_score(y_test, prediction))

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(staged_accuracy) + 1), staged_accuracy, marker="o")
plt.xlabel("Boosting Iteration")
plt.ylabel("Test Accuracy")
plt.title("Gradient Boosting: Accuracy vs Iteration")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 20: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_prediction = model.predict(new_sample)[0]
new_probability = model.predict_proba(new_sample)[0]

print("\n===================================")
print("NEW SAMPLE PREDICTION")
print("===================================")
print("Predicted Class:", target_names[new_prediction])
print("Probability of Class 0:", round(new_probability[0], 4))
print("Probability of Class 1:", round(new_probability[1], 4))

In [ ]:
#============================================================
# Experiment 12D: XGBoost Classification
# Real-World Dataset: Breast Cancer Wisconsin Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset Shape:")
print(X.shape)
print("\nTarget Classes:")
print(target_names)

# Step 2: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map({0: target_names[0], 1: target_names[1]})

print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nClass Distribution:")
print(df["Class"].value_counts())
print("\nDataset Description:")
print(df.describe())

# Step 4: Separate Features and Target
X = df[feature_names].values
y = df["Target"].values

# Step 5: Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 6: Create XGBoost Model
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

# Step 7: Train the Model
model.fit(X_train, y_train)

# Step 8: Make Predictions
y_pred = model.predict(X_test)

# Step 9: Prediction Probabilities
y_probability = model.predict_proba(X_test)

# Step 10: Display Actual and Predicted Values
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability_Class_0": y_probability[:, 0],
    "Probability_Class_1": y_probability[:, 1]
})
results["Actual Class"] = results["Actual"].map({0: target_names[0], 1: target_names[1]})
results["Predicted Class"] = results["Predicted"].map({0: target_names[0], 1: target_names[1]})

print("\nActual vs Predicted:")
print(results.head(10))

# Step 11: Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n===================================")
print("XGBOOST CLASSIFICATION PERFORMANCE")
print("===================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 12: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Step 13: Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Step 14: Feature Importance
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})
importance = importance.sort_values("Importance", ascending=False)

print("\nFeature Importance:")
print(importance)

# Step 15: Top 10 Important Features
top_features = importance.head(10)
print("\nTop 10 Important Features:")
print(top_features)

# Step 16: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

# Step 17: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names)
plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

# Step 18: Actual vs Predicted Plot
sample_number = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(sample_number, y_test, label="Actual", marker="o")
plt.scatter(sample_number, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number")
plt.ylabel("Class")
plt.title("XGBoost: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 19: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_prediction = model.predict(new_sample)[0]
new_probability = model.predict_proba(new_sample)[0]

print("\n===================================")
print("NEW SAMPLE PREDICTION")
print("===================================")
print("Predicted Class:", target_names[new_prediction])
print("Probability of Class 0:", round(new_probability[0], 4))
print("Probability of Class 1:", round(new_probability[1], 4))